# TIGER (Amazon Beauty) — full pipeline on Kaggle

Ye notebook `TIGER` reproduction repo ko end-to-end chalata hai:
content embeddings -> RQ-VAE -> Semantic IDs -> TIGER transformer -> evaluate.

**Pehle ye zaroor karein:**
1. `TIGER_Recommender_System_updated.zip` ko Kaggle par **Private Dataset** ke roop mein upload karein
   (Add Data -> Upload -> apni zip select karein).
2. Notebook Settings mein **Internet: ON** karein (Amazon reviews/meta download karne ke liye zaroori hai)
   aur **Accelerator: GPU (T4 x2 ya P100)** select karein.
3. Neeche `DATASET_SLUG` variable ko apne uploaded dataset ke naam se replace karein.

**Note on order:** README mein steps ka order thoda misleading hai — `dataloader_beauty.py`
`data/processed/*.jsonl` (sequences) ko pehle se maangta hai (item-filtering ke liye), isliye
hum pehle `preprocess_beauty` chalayenge, phir `dataloader_beauty`. Ye actual code dependency
follow karta hai, README ke listed order se nahi.


In [1]:
import os, shutil, glob

# Poore /kaggle/input ke andar TIGER_Recommender_System dhoondo — chahe kitni bhi nesting ho
src_candidates = glob.glob("/kaggle/input/**/TIGER_Recommender_System", recursive=True)
assert src_candidates, "Abhi bhi nahi mila — `!find /kaggle/input -maxdepth 6` chalakar check karein."
src = src_candidates[0]
dst = "/kaggle/working/tiger"

if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)
os.chdir(dst)
print("Working dir:", os.getcwd())
print(os.listdir("."))

Working dir: /kaggle/working/tiger
['configs', 'requirements.txt', 'README.md', 'tiger']


In [2]:
# Step 1: Install dependencies
!pip install -q -r requirements.txt

# torch.optim.Adafactor (native) needs torch>=2.5 — Kaggle usually ships a recent
# enough build, but upgrade explicitly to be safe.
import torch
print("torch:", torch.__version__)
if not hasattr(torch.optim, "Adafactor"):
    print("Upgrading torch for native Adafactor support...")
    !pip install -q -U torch --index-url https://download.pytorch.org/whl/cu121
    import importlib
    importlib.reload(torch)
    print("torch (after upgrade):", torch.__version__)

print("CUDA available:", torch.cuda.is_available())


torch: 2.10.0+cu128
CUDA available: True


## Stage A — Preprocess Amazon Beauty reviews -> leave-one-out sequences

Reviews dump download karta hai, 5-core filter apply karta hai, aur `data/processed/{train,val,test}.jsonl` likhta hai.
Expected stats: users=22,363 / items=12,101 / mean seq len=8.87 / median=6 (paper Table 6 se match).


In [3]:
!python -m tiger.scripts.preprocess_beauty --download

[prep] downloading http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Beauty_5.json.gz
[prep] wrote data/raw/reviews_Beauty_5.json.gz (44.8 MB)
[prep] parsed 198502 reviews
[prep] 5-core filter converged after 0 pass(es)
[prep] after 5-core: 198502 reviews
[prep] users=22363  items=12101  mean_seq_len=8.88  median_seq_len=6  (expected: 22,363 / 12,101 / 8.87 / 6)
[prep] wrote data/processed/train.jsonl (131413 rows)
[prep] wrote data/processed/val.jsonl (22363 rows)
[prep] wrote data/processed/test.jsonl (22363 rows)
[prep] done: train=131413  val=22363  test=22363


## Stage B — Item content embeddings (Sentence-T5)

Meta catalog download karta hai, Stage A ke sequence-items tak filter karta hai (~12k items),
aur Sentence-T5-XXL se content embeddings banata hai. `--download` sirf meta ke liye hai (agar already
maujood hai to skip ho jayega).

Note: `sentence-t5-xxl` bada model hai (~5GB) — pehli baar chalne mein time lagega.


In [4]:
import yaml

cfg_path = "configs/rqvae/beauty.yaml"
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

cfg["data"]["embedding_model"] = "sentence-transformers/sentence-t5-base"

with open(cfg_path, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(cfg["data"])

{'meta_path': 'data/amazon/meta_Beauty.json.gz', 'embedding_model': 'sentence-transformers/sentence-t5-base', 'input_dim': 768, 'standardize': True, 'sequences_dir': 'data/processed'}


In [5]:
!python -m tiger.rqvae.dataloader_beauty --config configs/rqvae/beauty.yaml --download

[data] downloading http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/meta_Beauty.json.gz
[data] wrote data/amazon/meta_Beauty.json.gz
[data] loaded 259204 beauty items
[data] filtered to 5-core items: 12101/259204 kept (12101 sequence items, 0 have no metadata)
[data] loading embedding model: sentence-transformers/sentence-t5-base
config_sentence_transformers.json: 100%|████████| 122/122 [00:00<00:00, 608kB/s]
README.md: 1.78kB [00:00, 6.88MB/s]
sentence_bert_config.json: 100%|██████████████| 53.0/53.0 [00:00<00:00, 216kB/s]
config.json: 1.39kB [00:00, 5.70MB/s]
model.safetensors: 100%|█████████████████████| 219M/219M [00:02<00:00, 77.8MB/s]
Loading weights: 100%|█| 99/99 [00:00<00:00, 1077.40it/s, Materializing param=en
tokenizer_config.json: 1.92kB [00:00, 4.66MB/s]
spiece.model: 100%|██████████████████████████| 792k/792k [00:00<00:00, 1.92MB/s]
tokenizer.json: 1.39MB [00:00, 42.5MB/s]
special_tokens_map.json: 1.79kB [00:00, 2.47MB/s]
config.json: 100%|█████████████████

## Stage C — Train RQ-VAE (paper hyperparameters)

encoder `[512,256,128]` -> latent `32` -> 3-level residual quantizer, codebook size `256` per level,
Adagrad lr=0.1, batch=1024, 20000 epochs, dead-code reinit enabled. Isse `outputs/amazon_beauty_checkpoints/best.pt` banega.

**Sanity check jab ye chal raha ho:** log mein `util=[...]` (codebook utilization per level) dekhte
rahein — paper mein ≥80% usage target hai. Agar kisi level ka usage bahut kam hai (<30-40%),
to codebook collapse ho raha hai — `reinit_every`/`reinit_usage_threshold` config check karein.


In [6]:
!python -m tiger.rqvae.train --config configs/rqvae/beauty.yaml

[train] device=cuda
[train] loaded embeddings (12101, 768)
[train] model L=3  K=256  D=32
[train] k-means init done on full corpus (12101 latents)
[train] epoch   1/20000  loss=6960441.3520  recon=6933717.3298  rq=26723.1680
[train] epoch   2/20000  loss=89.1722  recon=83.1474  rq=6.0248
[train] epoch   3/20000  loss=4.2007  recon=1.2918  rq=2.9089
[train] epoch   4/20000  loss=3.5507  recon=1.0908  rq=2.4599
[train] epoch   5/20000  loss=3.1951  recon=1.0298  rq=2.1653
[train] epoch   6/20000  loss=2.9314  recon=1.0126  rq=1.9187
[train] epoch   7/20000  loss=2.7228  recon=1.0056  rq=1.7172
[train] epoch   8/20000  loss=2.5563  recon=1.0024  rq=1.5539
[train] epoch   9/20000  loss=2.4348  recon=1.0008  rq=1.4340
[train] epoch  10/20000  loss=2.3420  recon=0.9999  rq=1.3420
[train] epoch  11/20000  loss=2.2678  recon=0.9993  rq=1.2685
[train] epoch  12/20000  loss=2.2071  recon=0.9989  rq=1.2082
[train] epoch  13/20000  loss=2.1555  recon=0.9986  rq=1.1569
[train] epoch  14/20000  loss

In [7]:
import os
ckpt_dir = "outputs/amazon_beauty_checkpoints"
print(os.listdir(ckpt_dir) if os.path.exists(ckpt_dir) else "Folder nahi mila")

['final.pt', 'best.pt']


In [8]:
import json
with open("outputs/amazon_beauty_metrics.json") as f:
    print(json.load(f))

{'recon_loss': 0.8667416558237314, 'utilization': [1.0, 1.0, 1.0], 'perplexity': [226.34254455566406, 190.7077178955078, 193.48373413085938], 'sid_unique_fraction': 0.8681927113461697, 'sid_collisions': 1595}


### (Optional but recommended) Inspect the trained Semantic IDs

Codebook utilization, perplexity, aur SID uniqueness fraction print karta hai — final sanity check
RQ-VAE training ke baad, retrieval-stage mein jaane se pehle.


In [9]:
!python -m tiger.rqvae.generate_sids --config configs/rqvae/beauty.yaml \
    --checkpoint outputs/amazon_beauty_checkpoints/best.pt

[gen] device=cuda
[gen] loaded outputs/amazon_beauty_checkpoints/best.pt  L=3 K=256
[gen] utilization per level: ['1.000', '1.000', '1.000']
[gen] perplexity per level:  ['226.03', '192.06', '193.75']
[gen] sid unique fraction:   0.8674  collisions=1604
[gen] wrote outputs/amazon_beauty_sids.csv  (12101 rows)

[sample] 10 items and their SIDs:
    80-183-197  {'item_id': '7806397051', 'title': 'WAWO 15 Color Professionl Makeup Eyeshadow Camouflage Facial Concealer Neutral Palette', 'brand': 'COKA', 'categories': 'Beauty, Makeup, Face, Concealers & Neutralizers', 'price': 5.04, 'description': 'An extensive range of 15 multiple vibrant long wear concealer colour with different skin tones to create more than 10,000 amazing looks. Using the most commonly applied shades, ensures the best skin colour match and guarantees a traceless and natural finish. Enabling layering and mixing, provides total camouflage for almost any skin problem including blemishes, scars, birthmarks and black circles.

## Stage D — Build item <-> Semantic-ID lookup tables

`(c0,c1,c2)` RQ-VAE se aata hai; collision-breaking `c3` yahan assign hota hai
(sequence-items ke subset par). Output: `data/item_to_sid.json`, `data/sid_to_item.json`.


In [10]:
!python -m tiger.scripts.build_sid_tables \
    --checkpoint outputs/amazon_beauty_checkpoints/best.pt \
    --items-csv  outputs/amazon_beauty_items.csv \
    --embeddings outputs/amazon_beauty_embeddings.npy \
    --sequences-dir data/processed \
    --output-dir   data

[sid] device=cuda
[sid] loaded 12101 items, embeddings (12101, 768)
[sid] 12101 unique items across train/val/test sequences
[sid] filtered to 12101 items present in both
[sid] encoded to codes, shape=(12101, 3)
[sid] max collision bucket size = 10 (c3 capacity = 256)
[sid] wrote data/item_to_sid.json  (12101 items)
[sid] wrote data/sid_to_item.json
[sid] 10497 unique (c0,c1,c2) prefixes across 12101 items  (avg collisions/prefix = 1.15)


## (Recommended) Quick smoke test before the full 200K-step run

4000 steps ka trimmed run — pipeline ke wiring ko verify karta hai (data loading, model forward,
beam-decode eval) bina paper-matching result expect kiye. Agar ye clean chal jaye bina error ke,
tabhi neeche wala full run start karein.


In [11]:
import sys
sys.path.insert(0, ".")
from tiger.retrieval.train import train_from_config

train_from_config(
    "configs/retrieval/beauty.yaml",
    data_dir="data",
    output_dir="outputs/tiger_beauty_smoketest",
    quick=True,
)

[train] device=cuda  amp_dtype=torch.bfloat16
[train] train=131413  val=22363
[train] model params: 4,847,104
[train] step=     50/4000  loss=5.8011  lr=1.00e-03  step_per_s=3.3
[train] step=    100/4000  loss=4.8666  lr=2.00e-03  step_per_s=3.3
[train] step=    150/4000  loss=4.1326  lr=3.00e-03  step_per_s=3.3
[train] step=    200/4000  loss=3.6939  lr=4.00e-03  step_per_s=3.3
[train] step=    250/4000  loss=3.4362  lr=5.00e-03  step_per_s=3.3
[train] step=    300/4000  loss=3.1956  lr=6.00e-03  step_per_s=3.4
[train] step=    350/4000  loss=3.0104  lr=7.00e-03  step_per_s=3.4
[train] step=    400/4000  loss=2.8969  lr=8.00e-03  step_per_s=3.4
[train] step=    450/4000  loss=2.7754  lr=9.00e-03  step_per_s=3.4
[train] step=    500/4000  loss=2.6844  lr=1.00e-02  step_per_s=3.4
[train] step=    550/4000  loss=2.5874  lr=9.53e-03  step_per_s=3.4
[train] step=    600/4000  loss=2.5518  lr=9.13e-03  step_per_s=3.4
[train] step=    650/4000  loss=2.4843  lr=8.77e-03  step_per_s=3.4
[train

{'model': {'vocab_size': 3027,
  'd_model': 128,
  'd_ff': 1024,
  'num_heads': 6,
  'head_dim': 64,
  'num_encoder_layers': 4,
  'num_decoder_layers': 4,
  'dropout': 0.1,
  'max_enc_len': 82,
  'max_dec_len': 5,
  'rel_num_buckets': 32,
  'rel_max_distance': 128,
  'tie_embeddings': True,
  'initializer_range': 0.02},
 'train': {'total_steps': 4000,
  'batch_size': 256,
  'warmup_steps': 500,
  'peak_lr': 0.01,
  'beta1': 0.9,
  'beta2': 0.999,
  'weight_decay': 0.0,
  'grad_clip': 1.0,
  'label_smoothing': 0.0,
  'seed': 42,
  'log_every': 50,
  'eval_every': 1000,
  'early_stop_patience': 5,
  'eval_batch_size': 64,
  'eval_beam_width': 50,
  'eval_top_k': 10,
  'eval_max_batches': 20,
  'num_workers': 2},
 'paths': {'data_dir': 'data', 'output_dir': 'outputs/tiger_beauty'}}

In [12]:
import yaml

cfg_path = "configs/retrieval/beauty.yaml"
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

cfg["train"]["total_steps"] = 200000

with open(cfg_path, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(cfg["train"])

{'total_steps': 200000, 'batch_size': 256, 'warmup_steps': 10000, 'peak_lr': 0.01, 'beta1': 0.9, 'beta2': 0.999, 'weight_decay': 0.0, 'grad_clip': 1.0, 'label_smoothing': 0.0, 'seed': 42, 'log_every': 50, 'eval_every': 5000, 'early_stop_patience': 5, 'eval_batch_size': 64, 'eval_beam_width': 50, 'eval_top_k': 10, 'eval_max_batches': None, 'num_workers': 2}


## Stage E — Train the TIGER transformer (full run, paper hyperparameters)

`d_model=128`, `6 heads`, `4+4 encoder/decoder layers`, batch=256, Adafactor peak_lr=0.01
with 10k-step warmup + inverse-sqrt decay, 200K total steps, early stop after 5 evals (25K steps)
without val NDCG@10 improvement.

**Time budget:** Isme kai ghante lag sakte hain (Kaggle GPU session ki single-run limit dhyan mein
rakhein — typically ~9-12h). Agar session cut ho jaye beech mein, is current codebase mein
resume-from-checkpoint logic nahi hai train loop mein — is limitation ko dhyan mein rakhkar,
zaroorat pade to `configs/retrieval/beauty.yaml` mein `total_steps` thoda kam karke pehle chalayein,
ya session timeout se pehle hi training complete ho jaye is hisaab se plan karein.


In [13]:
!python -m tiger.retrieval.train --config configs/retrieval/beauty.yaml

[train] device=cuda  amp_dtype=torch.bfloat16
[train] train=131413  val=22363
[train] model params: 4,847,104
[train] step=     50/200000  loss=8.0850  lr=5.00e-05  step_per_s=3.3
[train] step=    100/200000  loss=7.0485  lr=1.00e-04  step_per_s=3.3
[train] step=    150/200000  loss=6.3029  lr=1.50e-04  step_per_s=3.3
[train] step=    200/200000  loss=6.0015  lr=2.00e-04  step_per_s=3.3
[train] step=    250/200000  loss=5.6070  lr=2.50e-04  step_per_s=3.4
[train] step=    300/200000  loss=5.3076  lr=3.00e-04  step_per_s=3.4
[train] step=    350/200000  loss=5.0475  lr=3.50e-04  step_per_s=3.4
[train] step=    400/200000  loss=4.9083  lr=4.00e-04  step_per_s=3.4
[train] step=    450/200000  loss=4.7257  lr=4.50e-04  step_per_s=3.4
[train] step=    500/200000  loss=4.5332  lr=5.00e-04  step_per_s=3.4
[train] step=    550/200000  loss=4.4328  lr=5.50e-04  step_per_s=3.4
[train] step=    600/200000  loss=4.2177  lr=6.00e-04  step_per_s=3.4
[train] step=    650/200000  loss=4.0892  lr=6.50e

## Stage F — Evaluate the best checkpoint on the test split (vs. paper)

In [14]:
!python -m tiger.retrieval.evaluate --checkpoint outputs/tiger_beauty/checkpoints/best.pt

{
  "recall@5": 0.027992666457988643,
  "ndcg@5": 0.018532990056137105,
  "recall@10": 0.041854849528238605,
  "ndcg@10": 0.022958996466015343,
  "invalid_rate": 0.00022358359790725755
}
   metric   ours  paper  rel_%
 recall@5 0.0280 0.0454  -38.3
   ndcg@5 0.0185 0.0321  -42.3
recall@10 0.0419 0.0648  -35.4
  ndcg@10 0.0230 0.0384  -40.2


 Save everything for download

Checkpoints + metrics + SID tables ko ek zip mein daal kar `/kaggle/working/` mein rakhte hain
taaki "Output" tab se download ho sake.


In [15]:
!zip -r /kaggle/working/tiger_beauty_results.zip outputs data/item_to_sid.json data/sid_to_item.json data/processed
print("Saved: /kaggle/working/tiger_beauty_results.zip")

  adding: outputs/ (stored 0%)
  adding: outputs/tiger_beauty_smoketest/ (stored 0%)
  adding: outputs/tiger_beauty_smoketest/val_curve.png (deflated 12%)
  adding: outputs/tiger_beauty_smoketest/checkpoints/ (stored 0%)
  adding: outputs/tiger_beauty_smoketest/checkpoints/best.pt (deflated 8%)
  adding: outputs/tiger_beauty_smoketest/checkpoints/last.pt (deflated 8%)
  adding: outputs/tiger_beauty_smoketest/metrics.json (deflated 69%)
  adding: outputs/amazon_beauty_embeddings.npy (deflated 8%)
  adding: outputs/amazon_beauty_checkpoints/ (stored 0%)
  adding: outputs/amazon_beauty_checkpoints/final.pt (deflated 9%)
  adding: outputs/amazon_beauty_checkpoints/best.pt (deflated 9%)
  adding: outputs/amazon_beauty_metrics.json (deflated 53%)
  adding: outputs/amazon_beauty_sids.csv (deflated 65%)
  adding: outputs/amazon_beauty_items.csv (deflated 66%)
  adding: outputs/tiger_beauty/ (stored 0%)
  adding: outputs/tiger_beauty/val_curve.png (deflated 8%)
  adding: outputs/tiger_beauty/ch